# **SETUP**

In [2]:
import pathlib
import pandas as pd

PROJECT_ROOT = pathlib.Path().absolute().parent

train = pd.read_parquet(PROJECT_ROOT / "data" / "train.parquet")
test = pd.read_parquet(PROJECT_ROOT / "data" / "test.parquet")
cv = pd.read_parquet(PROJECT_ROOT / "data" / "cv.parquet")

# **FREQUENCY-ENCODING CATEGORICALS**

In [4]:
cat_cols = train.select_dtypes("category").columns
X_train, X_test = train[cat_cols], test[cat_cols]

oof_out = pd.concat([
    X_train.loc[cv.outer_fold.eq(k)].apply(
        lambda s: s.map(X_train.loc[~cv.outer_fold.eq(k), s.name].value_counts(normalize=True))
    )
    for k in sorted(cv.outer_fold.unique())
]).sort_index().fillna(0).astype("float64")

test_out = X_test.apply(
    lambda s: s.map(X_test[s.name].value_counts(normalize=True))
).fillna(0).astype("float32")

# **EXPORT**

In [9]:
oof_out.to_parquet(PROJECT_ROOT / "data" / "features" / "004-frequency-encode-categoricals" / "oof.parquet")
test_out.to_parquet(PROJECT_ROOT / "data" / "features" / "004-frequency-encode-categoricals" / "test.parquet")